# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates the process of loading, exploring, and processing the [FAIR<sup>2</sup> Croissant dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You'll load both the structured metadata and records, review available record sets and fields, extract and analyze data, and create visualizations.

### Dataset Source
The dataset source Croissant schema URL is:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

All entities (record sets, fields, and columns) are referenced by their `@id` fields. This ensures stable and reproducible data processing.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Show summary metadata (name and description)
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"License: {dataset.metadata.license}")
print(f"Date Published: {dataset.metadata.datePublished}")

## 2. Data Overview

Discover available **record sets** and their **fields**. Each entity below is referenced by its `@id` for reproducibility.

In [ ]:
# List all record sets in the dataset (using their '@id')
record_set_objs = dataset.metadata.record_sets

print(f"Total record sets: {len(record_set_objs)}\n")
for rs in record_set_objs:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name:        {rs.name}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    print(f"  Fields (@id and name):")
    for field in rs.fields:
        print(f"    Field @id: {field.id:50} Name: {field.name}")
    print("")

## 3. Data Extraction

Load the data from each record set into a Pandas DataFrame. Use the record set `@id` and field `@id` identified above.

For demonstration, we'll extract the main record set, which contains protein-level data. Replace the `main_record_set_id` variable with any other `@id` as needed.

In [ ]:
# Use each record set's @id for robust referencing
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Display them for selection
print("All available record_set @id values:")
for rid in record_set_ids:
    print(rid)

# In this dataset, the main record set is typically the one with 'protein' in its id or is first. Adjust below as needed.
main_record_set_id = record_set_ids[0]

# Load all records from all record sets into dataframes
dataframes = {}
for rs_id in record_set_ids:
    # .records yields dicts with field @id as keys
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"record_set @id: {rs_id}, records: {len(df)}")
    print(f"Columns (@id): {list(df.columns)}\n")

# Show a preview of the main table
df_main = dataframes[main_record_set_id]
print(f"Sample from record_set {main_record_set_id}:")
df_main.head()

## 4. Exploratory Data Analysis (EDA)

Conduct basic data processing and summarization—for example, filter the records based on a numeric field, normalize the field, and group by another field. All field references use their `@id` as columns.

The following code finds a numeric field (e.g., coverage, molecular weight), performs filtering, then normalizes and groups data. Modify the selected field `numeric_field_id` and `group_field_id` as desired!

In [ ]:
# List all field @id and names in this record set for selection
print("Available columns/field @id in the main record set:")
rs_obj = [rs for rs in dataset.metadata.record_sets if rs.id == main_record_set_id][0]
for field in rs_obj.fields:
    print(f"@id: {field.id:40} name: {field.name}")

# -- Replace with a numeric field in your actual dataset for demo purposes.
# For this dataset, molecular weight or coverage is likely numeric. Adjust below as needed --
# For example:
# numeric_field_id = 'cr:coverage'  # or 'cr:molecular_weight' or similar
# group_field_id = 'cr:sample'      # e.g., to group by sample or protein type

# Fill in an example from your field listing:
# Example (user will need to confirm actual @id is correct from list)
numeric_field_id = None
group_field_id = None
# Pick the first numeric-looking field if possible:
import numpy as np
for cid in df_main.columns:
    # Try to infer numeric: test if at least 80% of the data converts to float
    non_na = df_main[cid].dropna()
    try:
        arr_float = pd.to_numeric(non_na)
        if len(arr_float) > 0 and arr_float.dtype.kind in {'i', 'u', 'f'}:
            numeric_field_id = cid
            break
    except Exception:
        continue
if not numeric_field_id:
    print("No numeric field detected automatically. Please specify one from the overview above.")
else:
    print(f"Automatically selected numeric field: {numeric_field_id}")

# Pick a likely grouping field (string/categorical)
for cid in df_main.columns:
    if cid != numeric_field_id:
        nunique = df_main[cid].nunique()
        if 2 < nunique < len(df_main) // 5:
            group_field_id = cid
            break
if group_field_id:
    print(f"Automatically selected grouping field: {group_field_id}")
else:
    print("No grouping field detected automatically. Please specify one from the overview above.")

if numeric_field_id:
    # Convert numeric if not already
    series_num = pd.to_numeric(df_main[numeric_field_id], errors='coerce')
    threshold = series_num.quantile(0.75)
    filtered_df = df_main[series_num > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.4f} (top 25% values): {len(filtered_df)} records\n")

    # Normalize the selected numeric column
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    print(f"Normalized {numeric_field_id} (z-score).\n")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a field if one is found (categorical, not unique)
    if group_field_id:
        grouped_df = (
            filtered_df
            .groupby(group_field_id)[numeric_field_id]
            .mean()
            .sort_values(ascending=False)
        )
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id} (top 5 groups):")
        print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the main numeric field, and if available, group by the selected grouping field for a boxplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field_id:
    plt.figure(figsize=(7,4))
    pd.to_numeric(df_main[numeric_field_id], errors='coerce').hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id} (filtered)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook provided a workflow to programmatically access, inspect, and process a FAIR<sup>2</sup> dataset using `mlcroissant`, referencing entities by their `@id`. 

- We loaded the Croissant schema and examined the available record sets and fields (`@id`s).
- We extracted and explored the data, filtered and normalized a numeric field, and performed group-wise analysis.
- Visualizations summarized distributions and group-level patterns.

For further analysis or to integrate ML pipelines, continue working with the DataFrames created above. Always refer to fields and sets by their `@id` for reproducibility and future-proof code!